In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, lit, month, when, count
from pyspark.sql.types import DecimalType
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
#Iniciamos la sesion de spark
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Evidencia1") \
    .getOrCreate()

#Verificacion del entorno
spark.conf.set("spark.sql.eagerEval.enable",True)
spark

In [3]:
#Obtenemos todos los archivos que componen nuestra base datos
folder = r'C:\airline delay analysis'
archivos = os.listdir(folder)
archivos

['2009.csv',
 '2010.csv',
 '2011.csv',
 '2012.csv',
 '2013.csv',
 '2014.csv',
 '2015.csv',
 '2016.csv',
 '2017.csv',
 '2018.csv',
 '2019.csv']

In [4]:
#Precargamos una de las bases de datos
df = spark.read.csv(r"C:\airline delay analysis\2009.csv", header=True, inferSchema=True)

#Unimos todas las bases de datos
for fl in archivos[1:]:
    print(fl)
    dfTemp = spark.read.csv(folder+f'\\{fl}', header=True, inferSchema=True)
    df = df.unionByName(dfTemp,allowMissingColumns=True)

2010.csv
2011.csv
2012.csv
2013.csv
2014.csv
2015.csv
2016.csv
2017.csv
2018.csv
2019.csv


In [5]:
#Vista de tabla con Pandas solo para visualizacion
df.show(10)

+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+-----------+-----------------+----+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|Unnamed: 27|OP_UNIQUE_CARRIER|_c20|
+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+---

In [6]:
#Obtenemso el conteo de filas con las que cuenta el df
p = df.count()
print("Población (vuelos totales):", p)

Población (vuelos totales): 68979001


In [7]:
#Verificamos el esquema
df.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: double (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: double

In [8]:
#Creamos columna mes
df = df.withColumn("month", month("FL_DATE"))

#Verificamos los datos de nuevo
df.show(10) 

+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+-----------+-----------------+----+-----+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|Unnamed: 27|OP_UNIQUE_CARRIER|_c20|month|
+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+-

In [9]:
#Generamos la columna estacion en referencia a la estacion del año, creando asi la primer particion
df = df.withColumn(
    "season",
    when(col("month").isin(12, 1, 2), "Invierno")
    .when(col("month").isin(3, 4, 5), "Primavera")
    .when(col("month").isin(6, 7, 8), "Verano")
    .otherwise("Otoño")
)

#Verificamos la creacion de las estaciones
df.show(10)

+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+-----------+-----------------+----+-----+--------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|Unnamed: 27|OP_UNIQUE_CARRIER|_c20|month|  season|
+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-----------

In [10]:
#Realizamos una particion mas ahora verificando los retrasos registrados
tabla_retrasos = (
    df.groupBy("season").agg(
        count("DEP_DELAY").alias("vuelos_totales"), #Cantidad de vuelos por estacion
        sum(when(col("DEP_DELAY") > 0, 1).otherwise(0)).alias("vuelos_retrasados"), #Cantidad de vuelos retrasados
        sum(when(col("CARRIER_DELAY") > 0, 1).otherwise(0)).alias("retrasos_operacion"),#Cantidad de vuelos por motivo de operacion (mantenimiento, traslado, estacionamiento, etc.)
        sum(when(col("WEATHER_DELAY") > 0, 1).otherwise(0)).alias("retrasos_clima") #Cantidad de vuelos retrasados por clima
    )
)

tabla_retrasos.show(10)

+---------+--------------+-----------------+------------------+--------------+
|   season|vuelos_totales|vuelos_retrasados|retrasos_operacion|retrasos_clima|
+---------+--------------+-----------------+------------------+--------------+
| Invierno|      15921902|          6074316|           1618356|        207557|
|Primavera|      17291594|          6223481|           1570982|        167828|
|   Verano|      17967686|          7261065|           1954122|        243290|
|    Otoño|      16727034|          5266184|           1226581|        101033|
+---------+--------------+-----------------+------------------+--------------+



In [11]:
#Creamos las posibilidades de cada caso
tabla_probabilidades = (
    tabla_retrasos
    .withColumn("prob_temporada", (col("vuelos_totales") / lit(p)).cast(DecimalType(3,3))) #Probabilidad de que un vuelo ocurra en cierta estacion.
    .withColumn("prob_retraso_salida", (col("vuelos_retrasados") / col("vuelos_totales")).cast(DecimalType(3,3))) #Probalidad de sufrir un retraso en cierta estacion
    .withColumn("prob_retraso_operacion", (col("retrasos_operacion") / col("vuelos_totales")).cast(DecimalType(3,3))) #Probalidad de que retraso sea por operacion
    .withColumn("prob_retraso_clima", (col("retrasos_clima") / col("vuelos_totales")).cast(DecimalType(3,3))) #Probabilidad de que retraso sea por clima
    .select(
        "season",
        "prob_temporada",
        "prob_retraso_salida",
        "prob_retraso_operacion",
        "prob_retraso_clima"
    )
)

tabla_probabilidades.show(10)    


+---------+--------------+-------------------+----------------------+------------------+
|   season|prob_temporada|prob_retraso_salida|prob_retraso_operacion|prob_retraso_clima|
+---------+--------------+-------------------+----------------------+------------------+
| Invierno|         0.231|              0.382|                 0.102|             0.013|
|Primavera|         0.251|              0.360|                 0.091|             0.010|
|   Verano|         0.260|              0.404|                 0.109|             0.014|
|    Otoño|         0.242|              0.315|                 0.073|             0.006|
+---------+--------------+-------------------+----------------------+------------------+



# INICIO DE ACTIVIDAD 3

In [36]:
#Proporcion de vuelos con retraso total o DEP_DELAY > 0
df.filter("DEP_DELAY > 0").count()/df.count()

0.35989280273861896

## 2. Seleccion de datos

Se calculara una muestra con base en la formula de población infinita:
$$n = \frac{Z^2 \cdot p \cdot (1 - p)}{e^2}$$

Con valores:
* $Z = 1.96$ (nivel de confianza del 95%)
* $p  = 0.36$ (Coincide con lo observado en la columna prob_temporada)
* $e = 0.03$

De igual manera se comparará con un valor fijo como el ejemplo en el ejercicio anterior.

In [65]:
import math
Z=1.96 
prob=0.36
e=0.03
    
n = (Z**2 * prob * (1 - prob)) / (e**2)

print("Tamaño de muestra calculada:",math.ceil(n))

Tamaño de muestra calculada: 984


In [66]:
# Obtenemos el tamaño de muestras por estacion para evitar sesgos

prob_invierno = tabla_probabilidades.filter(tabla_probabilidades.season == "Invierno").select("prob_temporada").collect()[0][0]
prob_primavera = tabla_probabilidades.filter(tabla_probabilidades.season == "Primavera").select("prob_temporada").collect()[0][0]
prob_verano = tabla_probabilidades.filter(tabla_probabilidades.season == "Verano").select("prob_temporada").collect()[0][0]
prob_otono = tabla_probabilidades.filter(tabla_probabilidades.season == "Otoño").select("prob_temporada").collect()[0][0]
contInv = df.filter(df.season == "Invierno").count()
contPri = df.filter(df.season == "Primavera").count()
contVer = df.filter(df.season == "Verano").count()
contOto = df.filter(df.season == "Otoño").count()

fracciones = {
    "Invierno": (n*float(prob_invierno))/contInv,
    "Primavera": (n*float(prob_primavera))/contPri,
    "Verano": (n*float(prob_verano))/contVer,
    "Otoño": (n*float(prob_otono))/contOto
}

fracciones

{'Invierno': 1.3919634572370326e-05,
 'Primavera': 1.4071798080525443e-05,
 'Verano': 1.4022626223539718e-05,
 'Otoño': 1.4097616349042803e-05}

In [67]:
muestra = df.sampleBy(
    "season",
    fractions=fracciones,
    seed=143
)

print("La nueva instancia cuenta con",muestra.count(),"valores")
muestra.show(10)

La nueva instancia cuenta con 976 valores
+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+-----------+-----------------+----+-----+---------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|Unnamed: 27|OP_UNIQUE_CARRIER|_c20|month|   season|
+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-----

In [68]:
# Verificacion de vuelos con retraso total o DEP_DELAY > 0
muestra.filter("DEP_DELAY > 0").count()/muestra.count()

0.3637295081967213

Lo anterior demuestra que la muestra un mantiene una proporcion aceptable de la variable de salida, por lo que la muestra puede funcionar correctamente

In [69]:
muestra.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: double (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: double

## Preparacion de datos


In [70]:
#Eliminacion de columnas que no son significativas
eliminar = ['_c20','OP_UNIQUE_CARRIER','Unnamed: 27','CANCELLATION_CODE','OP_CARRIER_FL_NUM']
muestra = muestra.drop(*eliminar)


#Convertimos divert y cancelled a binario
muestra = muestra.withColumn("DIVERTED", col("DIVERTED").cast("byte"))
muestra = muestra.withColumn("CANCELLED", col("CANCELLED").cast("byte"))

# Con base en previas investigaciones se sabe que si no se tiene un evento de retraso no se registra ningun valor (null) por lo que se sustituiran en los campos con terminacion _DELAY por 0.0
colDelay = ['CARRIER_DELAY','WEATHER_DELAY','NAS_DELAY','SECURITY_DELAY','LATE_AIRCRAFT_DELAY','TAXI_OUT','TAXI_IN']
dictDelay = {k:0 for k in colDelay}
muestra = muestra.fillna(dictDelay)

#Los convertimos a enteros
for c in colDelay:
    muestra = muestra.withColumn(c, col(c).cast("int"))

muestra.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: double (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- CRS_ARR_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: byte (nullable = true)
 |-- DIVERTED: byte (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: integer (nullable = true)
 |-- WEATHER_DELAY: integer (nullable = true)
 |-- NAS_DELAY: integer (nullable = true)
 |-- SECURITY_DELAY: integer (nullable

In [71]:
# Ahora podemos visualizar datos para verificar valores anomalos mediante IQR y box plot
iqr_results = {}

for c in colDelay:
    q1, q3 = muestra.approxQuantile(c, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    iqr_results[c]={
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower": lower,
        "upper": upper
    }

iqr_results

{'CARRIER_DELAY': {'Q1': 0.0,
  'Q3': 0.0,
  'IQR': 0.0,
  'lower': 0.0,
  'upper': 0.0},
 'WEATHER_DELAY': {'Q1': 0.0,
  'Q3': 0.0,
  'IQR': 0.0,
  'lower': 0.0,
  'upper': 0.0},
 'NAS_DELAY': {'Q1': 0.0, 'Q3': 0.0, 'IQR': 0.0, 'lower': 0.0, 'upper': 0.0},
 'SECURITY_DELAY': {'Q1': 0.0,
  'Q3': 0.0,
  'IQR': 0.0,
  'lower': 0.0,
  'upper': 0.0},
 'LATE_AIRCRAFT_DELAY': {'Q1': 0.0,
  'Q3': 0.0,
  'IQR': 0.0,
  'lower': 0.0,
  'upper': 0.0},
 'TAXI_OUT': {'Q1': 10.0,
  'Q3': 18.0,
  'IQR': 8.0,
  'lower': -2.0,
  'upper': 30.0},
 'TAXI_IN': {'Q1': 4.0, 'Q3': 8.0, 'IQR': 4.0, 'lower': -2.0, 'upper': 14.0}}

Con los resultados de la columna anterior podemos ver que por si solos los campos "_DELAY" son tipicamente 0 o en otras palabras cualquier valor diferente de 0 es atipico. Pero se puede observar que "TAXI_OUT" que es el momento en que el avion deja la puerta de embarque si contiene valores tipicos rondando los 10 y 19 min, al igual que "TAXI_IN" que es cuando el avion se estaciona en la puerta de embarque. 

## Preparacion de conjunto de entrenamiento y prueba

In [72]:
# Aplicamos un muestreo aleatorio simple con 70% prueba y 30% test
train, test = muestra.randomSplit([0.7, 0.3], seed=143)

In [73]:
# Verificacion de vuelos con retraso total o DEP_DELAY > 0 de los conjuntos
print('Proporcion de DEP_DELAY en train:',train.filter("DEP_DELAY > 0").count()/train.count())
print('Proporcion de DEP_DELAY en test:',test.filter("DEP_DELAY > 0").count()/test.count())

Proporcion de DEP_DELAY en train: 0.36617647058823527
Proporcion de DEP_DELAY en test: 0.3581081081081081


Podemos observar que la aplicacion del muestreo aleatorio simple sigue mantienendo la proporcion de la poblacion.


## Modelos de aprendizaje supervisado y no supervisado

### Modelo no supervisado

#### **K-Means para muestra**

In [89]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

trainModel = train.select(
    "DEP_DELAY",
    "TAXI_OUT",
    "DISTANCE",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "month"
).na.fill(0)

assembler = VectorAssembler(
    inputCols=trainModel.columns,
    outputCol="features"
)

train_vec = assembler.transform(trainModel)

In [90]:
kmeans = KMeans(k=4, seed=143, featuresCol="features", predictionCol="cluster")
modelo = kmeans.fit(train_vec)

In [91]:
# Test
testModel = test.select(
    "DEP_DELAY",
    "TAXI_OUT",
    "DISTANCE",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "month"
).na.fill(0)

assembler = VectorAssembler(
    inputCols=testModel.columns,
    outputCol="features"
)

test_vec = assembler.transform(testModel)

In [92]:
predicciones = modelo.transform(test_vec)
predicciones.select("cluster").show(10)

+-------+
|cluster|
+-------+
|      1|
|      1|
|      0|
|      0|
|      1|
|      2|
|      1|
|      1|
|      1|
|      1|
+-------+
only showing top 10 rows


In [93]:
centros = modelo.clusterCenters()
for i, c in enumerate(centros):
    print(f"Cluster {i}: {c}")

Cluster 0: [8.27272727e+00 1.58133971e+01 8.79526316e+02 3.49282297e+00
 4.88038278e-01 1.99043062e+00 4.66507177e+00 6.14832536e+00]
Cluster 1: [ 10.8252149   14.88538682 355.72492837   5.67908309   0.40114613
   2.89111748   3.69627507   6.43839542]
Cluster 2: [1.80227273e+01 1.88409091e+01 2.33495455e+03 2.63636364e+00
 0.00000000e+00 5.22727273e-01 1.45000000e+01 6.02272727e+00]
Cluster 3: [8.42307692e+00 1.61794872e+01 1.56106410e+03 2.25641026e+00
 5.25641026e-01 1.51282051e+00 4.34615385e+00 6.29487179e+00]


In [95]:
predicciones.groupBy("cluster", "month").count().show()

+-------+-----+-----+
|cluster|month|count|
+-------+-----+-----+
|      1|    1|   12|
|      0|    1|   11|
|      0|    2|    7|
|      1|    3|    7|
|      2|    3|    2|
|      1|    4|   13|
|      3|    5|    5|
|      1|    6|   13|
|      0|    6|   11|
|      0|    7|    6|
|      1|    7|   15|
|      3|    7|    5|
|      1|    8|   15|
|      1|    9|   12|
|      3|    9|    4|
|      1|   11|   16|
|      0|   11|    6|
|      1|   12|   14|
|      2|   12|    5|
|      0|   12|    5|
+-------+-----+-----+
only showing top 20 rows


In [97]:
print(predicciones.withColumn('cluster', predicciones.cluster == 0).show(1))
print(predicciones.withColumn('cluster', predicciones.cluster == 1).show(1))
print(predicciones.withColumn('cluster', predicciones.cluster == 2).show(1))
print(predicciones.withColumn('cluster', predicciones.cluster == 3).show(1))

+---------+--------+--------+-------------+-------------+---------+-------------------+-----+--------------------+-------+
|DEP_DELAY|TAXI_OUT|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|LATE_AIRCRAFT_DELAY|month|            features|cluster|
+---------+--------+--------+-------------+-------------+---------+-------------------+-----+--------------------+-------+
|     -8.0|      13|   227.0|            0|            0|        0|                  0|    1|[-8.0,13.0,227.0,...|  false|
+---------+--------+--------+-------------+-------------+---------+-------------------+-----+--------------------+-------+
only showing top 1 row
None
+---------+--------+--------+-------------+-------------+---------+-------------------+-----+--------------------+-------+
|DEP_DELAY|TAXI_OUT|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|LATE_AIRCRAFT_DELAY|month|            features|cluster|
+---------+--------+--------+-------------+-------------+---------+-------------------+-----+------------------